# Cybersecurity & Networking RAG Chatbot (Colab, T4 GPU)

This notebook uses your Q&A dataset (CSV) to build a simple **RAG (Retrieval-Augmented Generation)** chatbot.

**Before running:** go to Runtime > Change runtime type > Hardware accelerator and select **T4 GPU**.

Steps:
1. Install packages
2. Upload your `cybersecurity_networking_qa_dataset.csv`
3. Build embeddings + a FAISS vector store
4. Retrieval function
5. Load the chat LLM (Qwen2.5-3B-Instruct, fits on a T4)
6. RAG answer function
7. Gradio chat interface (with a shareable link)


In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate gradio pandas datasets pyarrow huggingface_hub

## Step 1: Upload your dataset
Run the cell below and browse/upload your `cybersecurity_networking_qa_dataset.csv` file.

In [ ]:
import os, re, json, hashlib
import numpy as np
import pandas as pd
import torch

# ---------------- Settings ----------------
USE_LOCAL_CSV = True     # also include your own question/answer CSV? (False = HF datasets only)
REBUILD_INDEX = False    # set True to re-create embeddings (use this when the data changes)
USE_DRIVE     = True     # save/load the index on Google Drive

EMBED_MODEL   = "BAAI/bge-small-en-v1.5"
RERANK_MODEL  = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_NAME      = "Qwen/Qwen2.5-3B-Instruct"
SAVE_DIR      = "/content/drive/MyDrive/rag_chatbot"

# kind: "alpaca"   -> instruction / input / output
#       "commands" -> command / description / usage / category / risk
#       "auto"     -> columns are auto-detected (negatives are always ignored)
HF_DATASETS = {
    "cyber_qa":   {"path": "mariiazhiv/cybersecurity_qa",                "kind": "alpaca"},
    "cyber_neg":  {"path": "jobby32/cybersecurity-QA-with-negatives",    "kind": "auto"},
    "net_alpaca": {"path": "beyazitkelceoglu/networking_dataset_alpaca", "kind": "alpaca"},
    "net_cmds":   {"path": "darkknight25/Networking_Commands_Dataset",   "kind": "commands"},
}

In [ ]:
def to_text(x):
    if x is None: return ""
    if hasattr(x, "tolist"): x = x.tolist()          # numpy array / scalar
    if isinstance(x, float) and x != x: return ""     # NaN
    if isinstance(x, (list, tuple)):
        return "\n".join(t for t in (to_text(i) for i in x) if t)
    if isinstance(x, dict): return json.dumps(x, ensure_ascii=False)
    return re.sub(r"[ \t]+", " ", str(x)).strip()

def find_col(cols, cands):
    low = {c.lower(): c for c in cols}
    for c in cands:
        if c in low: return low[c]
    return None

def is_neg_col(c):
    return "neg" in c.lower()

Q_CANDS = ["question", "query", "anchor", "instruction", "prompt", "user", "q", "input", "text"]
A_CANDS = ["answer", "positive", "pos", "output", "response", "completion", "assistant", "a", "context", "passage"]
ID_LIKE = {"id", "idx", "index", "uuid", "_id"}

def rec(q, a, source, search_text=""):
    return {"question": to_text(q), "answer": to_text(a), "source": source, "search_text": to_text(search_text)}

# ---------------------------------------------------------------- generic / auto
def adapt_generic(rows, cols, source):
    cols = list(cols)
    out = []

    # chat style: messages / conversations = [{role, content}, ...]
    chat = find_col(cols, ["messages", "conversations", "conversation"])
    if chat and not find_col(cols, ["question", "query", "anchor", "instruction", "prompt"]):
        for r in rows:
            msgs = r[chat]
            if hasattr(msgs, "tolist"): msgs = msgs.tolist()
            u = next((m for m in msgs if (m.get("role") or m.get("from")) in ("user", "human")), None)
            a = next((m for m in msgs if (m.get("role") or m.get("from")) in ("assistant", "gpt")), None)
            if u and a:
                out.append(rec(u.get("content") or u.get("value"), a.get("content") or a.get("value"), source))
        return out

    q_col = find_col([c for c in cols if not is_neg_col(c)], Q_CANDS)
    a_col = find_col([c for c in cols if not is_neg_col(c)], A_CANDS)

    if q_col is None or a_col is None or q_col == a_col:
        # fallback: guess from the text columns (skipping id and negative columns)
        text_cols = [c for c in cols
                     if not is_neg_col(c) and c.lower() not in ID_LIKE
                     and isinstance(rows[0].get(c), str)]
        if len(text_cols) < 2:
            raise ValueError(f"[{source}] could not detect question/answer columns. Columns: {cols}")
        q_col = text_cols[0]
        a_col = max(text_cols[1:], key=lambda c: np.mean([len(to_text(r.get(c))) for r in rows[:200]]))
        print(f"   ⚠️ [{source}] guessed columns: question='{q_col}', answer='{a_col}' — please check the preview")

    # if there is a label column (0/1), keep only the positive rows
    lab = find_col(cols, ["label", "is_positive", "relevant"])
    keep_label = None
    if lab:
        try:
            vals = {r.get(lab) for r in rows}
            if vals <= {0, 1, True, False}:
                keep_label = lab
        except TypeError:
            pass

    for r in rows:
        if keep_label and not r.get(keep_label): continue
        out.append(rec(r[q_col], r[a_col], source))
    print(f"   mapping [{source}]: question <- '{q_col}', answer <- '{a_col}'"
          + (f", label filter '{keep_label}'==1" if keep_label else "")
          + f"  | ignored negative cols: {[c for c in cols if is_neg_col(c)]}")
    return out

# ---------------------------------------------------------------- alpaca
def adapt_alpaca(rows, cols, source):
    ins = find_col(cols, ["instruction"])
    out_c = find_col(cols, ["output", "response"])
    inp = find_col(cols, ["input"])
    if not (ins and out_c):
        print(f"   ⚠️ [{source}] alpaca columns not found, falling back to auto-detect")
        return adapt_generic(rows, cols, source)
    res = []
    for r in rows:
        q = to_text(r[ins])
        extra = to_text(r[inp]) if inp else ""
        if extra: q = f"{q}\n{extra}"
        res.append(rec(q, r[out_c], source))
    print(f"   mapping [{source}]: question <- '{ins}'" + (f" + '{inp}'" if inp else "") + f", answer <- '{out_c}'")
    return res

# ---------------------------------------------------------------- commands
def adapt_commands(rows, cols, source):
    cmd = find_col(cols, ["command", "cmd"])
    desc = find_col(cols, ["description", "desc", "explanation"])
    if not (cmd and desc):
        print(f"   ⚠️ [{source}] command/description columns not found, falling back to auto-detect")
        return adapt_generic(rows, cols, source)
    usage = find_col(cols, ["usage", "syntax", "example"])
    cat   = find_col(cols, ["category", "type"])
    risk  = find_col(cols, ["risk", "warning", "caution"])
    res = []
    for r in rows:
        c, d = to_text(r[cmd]), to_text(r[desc])
        if not c or not d: continue
        u = to_text(r[usage]) if usage else ""
        ct = to_text(r[cat]) if cat else ""
        rk = to_text(r[risk]) if risk else ""
        q = f"What does the networking command `{c}` do?"
        parts = [f"Command: `{c}`", f"Purpose: {d}"]
        if u:  parts.append(f"Usage: {u}")
        if ct: parts.append(f"Category: {ct}")
        if rk: parts.append(f"Risk / caution: {rk}")
        res.append(rec(q, "\n".join(parts), source, search_text=f"{c} {d} {u} {ct}"))
    print(f"   mapping [{source}]: command='{cmd}', description='{desc}', usage='{usage}', category='{cat}', risk='{risk}'")
    return res

ADAPTERS = {"alpaca": adapt_alpaca, "commands": adapt_commands, "auto": adapt_generic}

# ---------------------------------------------------------------- loader (all splits + fallback)
def load_hf(path):
    from datasets import load_dataset
    try:
        dd = load_dataset(path)   # DatasetDict -> all splits (train, validation, test...)
        frames = [dd[k].to_pandas().assign(_split=k) for k in dd.keys()]
        return pd.concat(frames, ignore_index=True)
    except Exception as e:
        print(f"   ℹ️ load_dataset failed ({type(e).__name__}), trying the raw files instead...")
        from huggingface_hub import list_repo_files, hf_hub_download
        files = [f for f in list_repo_files(path, repo_type="dataset")
                 if f.lower().endswith((".jsonl", ".json", ".csv", ".parquet"))]
        if not files: raise
        frames = []
        for f in files:
            p = hf_hub_download(path, f, repo_type="dataset")
            fl = f.lower()
            if fl.endswith(".csv"):       frames.append(pd.read_csv(p))
            elif fl.endswith(".parquet"): frames.append(pd.read_parquet(p))
            else:
                try:    frames.append(pd.read_json(p, lines=fl.endswith(".jsonl")))
                except ValueError: frames.append(pd.read_json(p, lines=True))
        return pd.concat(frames, ignore_index=True)

## Step 2: Build embeddings + FAISS vector store

In [ ]:
local_records = []
if USE_LOCAL_CSV:
    try:
        from google.colab import files
        uploaded = files.upload()
    except Exception:
        uploaded = {}
    for fn in uploaded:
        if fn.lower().endswith(".csv"):
            d = pd.read_csv(fn)
            d.columns = [c.strip().lower() for c in d.columns]
            if {"question", "answer"} <= set(d.columns):
                local_records += [rec(r["question"], r["answer"], "local_csv") for r in d.to_dict("records")]
            else:
                print(f"⚠️ {fn}: 'question' and 'answer' columns not found, skipping. Columns: {list(d.columns)}")
print("Local CSV records:", len(local_records))

In [ ]:
loaded = {}
for name, cfg in HF_DATASETS.items():
    print(f"\n=== {name}  ({cfg['path']}) ===")
    try:
        raw_df = load_hf(cfg["path"])
    except Exception as e:
        print(f"❌ failed to load: {type(e).__name__}: {e}")
        continue
    loaded[name] = raw_df
    print(f"✅ rows={len(raw_df)} | columns={list(raw_df.columns)}")
    first = {k: (to_text(v)[:120] + ("..." if len(to_text(v)) > 120 else "")) for k, v in raw_df.iloc[0].to_dict().items()}
    print("   sample:", first)

In [ ]:
records = list(local_records)
for name, raw_df in loaded.items():
    print(f"\n--- adapting {name} ---")
    rows = raw_df.to_dict("records")
    try:
        recs = ADAPTERS[HF_DATASETS[name]["kind"]](rows, list(raw_df.columns), name)
    except Exception as e:
        print(f"❌ failed to adapt {name}: {e}")
        continue
    print(f"   -> {len(recs)} pairs")
    records += recs

df = pd.DataFrame(records)
before = len(df)

# clean
df["question"] = df["question"].str.strip()
df["answer"]   = df["answer"].str.strip()
df = df[(df["question"].str.len() >= 5) & (df["answer"].str.len() >= 10)]
df = df[df["question"].str.lower() != df["answer"].str.lower()]

# remove exact duplicates (same question AND same answer). Same question with a different answer is kept.
key = (df["question"].str.lower().str.replace(r"\s+", " ", regex=True) + "||" +
       df["answer"].str.lower().str.replace(r"\s+", " ", regex=True))
df = df[~key.duplicated()].reset_index(drop=True)

print(f"\nTotal: {before} -> {len(df)} (after cleaning + dedupe)")
print(df["source"].value_counts().to_string())

missing = [n for n in HF_DATASETS if n not in set(df["source"])]
if missing:
    print("\n⚠️ These datasets did not make it into the DB:", missing, "-> check the error/mapping messages above")

print("\n===== Preview (2 rows per source) =====")
for src, g in df.groupby("source"):
    for _, r in g.head(2).iterrows():
        print(f"[{src}] Q: {r['question'][:150]!r}\n         A: {r['answer'][:200]!r}\n")

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer(EMBED_MODEL, device=device)
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # query prefix for BGE

questions = df["question"].tolist()
answers   = df["answer"].tolist()
sources   = df["source"].tolist()

# Text to embed: search_text (for commands) or the question + the first part of the answer
texts = [(st if st else f"{q} {a[:300]}") for q, a, st in zip(questions, answers, df["search_text"])]
data_hash = hashlib.md5(("\n".join(texts) + EMBED_MODEL).encode()).hexdigest()

INDEX_PATH, DF_PATH, HASH_PATH = None, None, None
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(SAVE_DIR, exist_ok=True)
        INDEX_PATH = f"{SAVE_DIR}/faiss.index"
        DF_PATH    = f"{SAVE_DIR}/data.parquet"
        HASH_PATH  = f"{SAVE_DIR}/data.hash"
    except Exception as e:
        print("Drive mount failed, continuing without cache:", e)

index = None
if INDEX_PATH and not REBUILD_INDEX and os.path.exists(INDEX_PATH) and os.path.exists(HASH_PATH):
    if open(HASH_PATH).read().strip() == data_hash:
        index = faiss.read_index(INDEX_PATH)
        print(f"✅ Loaded cached index from Drive ({index.ntotal} entries)")

if index is None:
    emb = embedder.encode(texts, batch_size=128, convert_to_numpy=True,
                          normalize_embeddings=True, show_progress_bar=True).astype("float32")
    index = faiss.IndexFlatIP(emb.shape[1])     # normalized vectors + inner product = cosine similarity
    index.add(emb)
    if INDEX_PATH:
        faiss.write_index(index, INDEX_PATH)
        df.to_parquet(DF_PATH)
        open(HASH_PATH, "w").write(data_hash)
        print("💾 Index saved to Drive")

assert index.ntotal == len(df), "Index size does not match the data -> set REBUILD_INDEX=True"
print(f"FAISS ready: {index.ntotal} entries")

In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder(RERANK_MODEL, device=device, max_length=512)

def retrieve(query, top_k=4, fetch_k=20, min_score=0.40, source_filter=None):
    q_emb = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q_emb, fetch_k)

    cands = [(int(i), float(s)) for i, s in zip(idxs[0], scores[0])
             if i != -1 and s >= min_score and (source_filter is None or sources[i] in source_filter)]
    if not cands:
        return []

    rr = reranker.predict([(query, f"{questions[i]} {answers[i][:400]}") for i, _ in cands])
    ranked = sorted(zip(cands, rr), key=lambda x: x[1], reverse=True)[:top_k]

    return [{"question": questions[i], "answer": answers[i], "source": sources[i],
             "score": round(s, 3), "rerank": round(float(r), 2)} for (i, s), r in ranked]

def show(query, **kw):
    print(f"\n🔎 {query}")
    res = retrieve(query, **kw)
    if not res: print("   (no relevant match found)")
    for r in res:
        print(f"   [{r['source']} | sim={r['score']} | rr={r['rerank']}] {r['question'][:90]}\n      -> {r['answer'][:140]!r}")

# quick tests
show("what is malware")
show("what does the command ip pim bidir-enable do")
show("difference between TCP and UDP")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
model = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16, device_map="auto")
print("Model loaded on:", model.device)

In [ ]:
SHOW_SOURCES = True

SYSTEM_PROMPT = (
    "You are a helpful assistant for cybersecurity and networking questions. "
    "Explain in simple, clear language for beginners. "
    "Use the reference material below as your primary knowledge base and stay consistent with it. "
    "do NOT list any dataset sources."
    "If the reference is missing or not relevant, use your general knowledge and mention briefly that it is not from the knowledge base. "
    "Never invent commands, flags, or facts. Keep answers under about 250 words. "
    "Give defensive and educational guidance only."

)

def _history_to_messages(history, n=3):
    msgs = []
    for turn in (history or [])[-n * 2:]:
        if isinstance(turn, dict):                       # gradio messages format
            c = turn.get("content")
            if isinstance(c, list):                      # multimodal-style content
                c = " ".join(x.get("text", "") if isinstance(x, dict) else str(x) for x in c)
            if turn.get("role") in ("user", "assistant") and c:
                msgs.append({"role": turn["role"], "content": c})
        else:                                            # old [user, bot] tuples
            msgs += [{"role": "user", "content": turn[0]}, {"role": "assistant", "content": turn[1]}]
    return msgs

def generate_answer(query, history=None, top_k=4, max_new_tokens=600):
    retrieved = retrieve(query, top_k=top_k)

    if retrieved:
        context = "\n\n".join(
            f"[Source: {r['source']}]\nQ: {r['question']}\nA: {r['answer'][:1200]}" for r in retrieved)
    else:
        context = "No relevant reference found in the knowledge base."

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages += _history_to_messages(history)
    messages.append({"role": "user", "content": f"Reference:\n{context}\n\nUser question: {query}"})

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.4, top_p=0.9,
                                do_sample=True, pad_token_id=tokenizer.eos_token_id)

    answer = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


    return answer

print(generate_answer("what is cyber security?"))

In [ ]:
import gradio as gr

def chat_fn(message, history):
    return generate_answer(message, history=history)

demo = gr.ChatInterface(
    fn=chat_fn,
    title="Cybersecurity & Networking Chatbot",
    description="Ask questions about cybersecurity and networking (TCP/IP, OSI, malware, encryption, Cisco commands, etc.)",
    examples=["What is a firewall?", "Difference between TCP and UDP", "What does the command ip pim bidir-enable do?"],
)
demo.launch(share=True, debug=True)